In [5]:
import soundfile as sf
from IPython.display import Audio, display, Markdown
from hrtf import HRTF
from Trajectory import CircularTrajectory
from DynamicConvolver import DynamicConvolver
import numpy as np

# 1) Charger HRTF + signal
hrtf = HRTF.from_sofa("dataset/IRC_1002_C_44100.sofa")
signal, sr = sf.read("impulse_repeated.wav")
if signal.ndim == 2:
    signal = signal.mean(axis=1)

# 2) Definir une trajectoire (exemple)
traj = CircularTrajectory(
    duration_s=len(signal) / sr,
    period_s=3,
    elevation=0,
    start_az=0,
    clockwise=True,
 )

# 3) Convolution dynamique
conv = DynamicConvolver(
    hrtf=hrtf,
    signal=signal,
    sr=sr,
    trajectory=traj,
    segment_ms=50,
    overlap_ms=10
 )

output = conv.run()  # np.ndarray shape (N, 2)

# 4) Sauvegarde (optionnel)
# conv.save("Left_dynamic.wav", "Right_dynamic.wav", "Merged_dynamic.wav")

# 5) Ecoute des 3 sorties
left = output[:, 0]
right = output[:, 1]

# Versions stereo pour ecouter chaque oreille separement au casque
left_only = np.stack([left, np.zeros_like(left)], axis=0)
right_only = np.stack([np.zeros_like(right), right], axis=0)
merged = output.T

display(Markdown("### Left (oreille gauche seule)"))
display(Audio(left_only, rate=sr, normalize=False))

display(Markdown("### Right (oreille droite seule)"))
display(Audio(right_only, rate=sr, normalize=False))

display(Markdown("### Merged (binaural stereo)"))
display(Audio(merged, rate=sr, normalize=False))

[SegmentEngine] Démarrage : 200 segments × 50 ms  |  overlap=10 ms  |  HRIR=512 smp  |  crossfade='cosine'
[SegmentEngine] Rendu terminé : 441511 échantillons (10.01s)


### Left (oreille gauche seule)

### Right (oreille droite seule)

### Merged (binaural stereo)